# NetworkX Solver Comparison

This notebook compares different NetworkX pathfinding solvers on elevation data.
For a random tensor file at a given resolution (default 4096), we:
1. Load the elevation data
2. Generate random start/end markers
3. Run each solver with configurable cost models
4. Compare execution times and paths

Notes:
- Solver names use the `networkx:` prefix in config (e.g., `networkx:astar`).
- `cost_model` supports elevation, distance, slope_uphill, energy, absolute_elevation.
- Graph construction uses the cached edges backend (no configurable backend).

In [1]:
import sys
import time
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

# Add src to path for imports
sys.path.insert(0, str(Path("../src").resolve()))

from passage.generate import find_path

## Configuration

Set the resolution and optionally specify a tensor index. Configure cost model for the solver comparison.

In [2]:
# Configuration
RESOLUTION = 128  # Resolution to test (64, 128, 256, 512, 1024, 2048, 4096)
TENSOR_INDEX = 3  # Set to integer to use specific tensor, or None for random

# Cost model
COST_MODEL = "elevation"  # elevation, distance, slope_uphill, energy, absolute_elevation

# Solver configurations to compare
# Edge weights account for physical distance: 30m cardinal, 42.43m diagonal
SOLVERS = [
    {"name": "networkx:astar", "heuristic": "euclidean", "label": "A* (Euclidean)"},
    {"name": "networkx:astar", "heuristic": "manhattan", "label": "A* (Manhattan)"},
    {"name": "networkx:dijkstra", "label": "Dijkstra"},
    {"name": "networkx:dijkstra_bi", "label": "Dijkstra (Bidirectional)"},
    {"name": "networkx:bellman_ford", "label": "Bellman-Ford"},
    {"name": "networkx:unweighted", "label": "Unweighted (BFS)"},
]

# Common solver parameters
WEIGHT = 30.0
DIAGONAL_MOVEMENT = "always"

## Load Tensor Data

Find available tensors at the specified resolution and load one.

In [3]:
# Find tensor files
tensors_dir = Path(f"../outputs/dataset/{RESOLUTION}/tensors")
metadata_dir = Path(f"../outputs/dataset/{RESOLUTION}/metadata")

if not tensors_dir.exists():
    raise FileNotFoundError(
        f"No tensors found at {tensors_dir}. Generate samples first with: uv run passage generate --resolution {RESOLUTION}"
    )

tensor_files = sorted(tensors_dir.glob("*.npy"))
print(f"Found {len(tensor_files)} tensor files at resolution {RESOLUTION}")

# Select tensor
if TENSOR_INDEX is not None:
    tensor_path = tensors_dir / f"{TENSOR_INDEX:010d}.npy"
else:
    tensor_path = random.choice(tensor_files)

print(f"Using tensor: {tensor_path.name}")

# Load tensor (shape: H, W, 2) - channels: elevation, markers
tensor = np.load(tensor_path)
print(f"Tensor shape: {tensor.shape}")

# Extract elevation channel (normalized 0-1) and markers
elevation = tensor[:, :, 0]
markers = tensor[:, :, 1]

# Load metadata for reference
metadata_path = metadata_dir / tensor_path.with_suffix(".yaml").name
with open(metadata_path) as f:
    metadata = yaml.safe_load(f)

crop_min = float(metadata["crop_min_elevation"])
crop_max = float(metadata["crop_max_elevation"])
elevation_raw = elevation * (crop_max - crop_min) + crop_min

print("\nMetadata:")
print(f"  Target elevation: {metadata['target_elevation']:.1f}m")
print(f"  Crop elevation range: {crop_min:.1f}m - {crop_max:.1f}m")
print(f"  Solver: {metadata.get('solver', {}).get('name', 'N/A')}")

Found 10 tensor files at resolution 128
Using tensor: 0000000003.npy
Tensor shape: (128, 128, 2)

Metadata:
  Target elevation: 3859.5m
  Crop elevation range: 3847.0m - 5258.0m
  Solver: dijkstra_bi


## Extract Markers

Get start and end positions from the tensor's marker channel.

In [4]:
# Find marker positions
marker_positions = np.argwhere(markers > 0.5)
print(f"Found {len(marker_positions)} markers")

if len(marker_positions) >= 2:
    start = tuple(marker_positions[0])
    end = tuple(marker_positions[1])
else:
    # Generate random markers if not found
    h, w = elevation.shape
    start = (random.randint(0, h - 1), random.randint(0, w - 1))
    end = (random.randint(0, h - 1), random.randint(0, w - 1))

print(f"Start: {start}")
print(f"End: {end}")
print(f"Euclidean distance: {np.sqrt((start[0] - end[0]) ** 2 + (start[1] - end[1]) ** 2):.1f} pixels")

Found 2 markers
Start: (np.int64(0), np.int64(17))
End: (np.int64(53), np.int64(69))
Euclidean distance: 74.2 pixels


## Run Solver Comparison

Test each solver and record timing and path length.

In [5]:
# Use raw elevation in meters for cost calculations
elevation_input = elevation_raw

results = []

for solver_info in SOLVERS:
    solver_config = {
        "name": solver_info["name"],
        "diagonal_movement": DIAGONAL_MOVEMENT,
        "weight": WEIGHT,
        "cost_model": COST_MODEL,
    }
    if "heuristic" in solver_info:
        solver_config["heuristic"] = solver_info["heuristic"]

    print(f"\nRunning {solver_info['label']}...")
    print(f"  name={solver_config['name']}, cost_model={COST_MODEL}")

    # Time the full find_path call (includes graph construction)
    t0 = time.time()
    path = find_path(elevation_input, start, end, solver_config)
    total_time = time.time() - t0

    results.append(
        {
            "solver": solver_info["label"],
            "path_length": len(path),
            "time_ms": total_time * 1000,
            "cost_model": COST_MODEL,
            "backend": "networkx",
            "path": path,
        }
    )

    print(f"  Path length: {len(path)} pixels")
    print(f"  Time: {total_time * 1000:.1f} ms")

print("\n" + "=" * 50)
print("Comparison complete!")

2026-01-15 14:01:05.391 | DEBUG    | passage.generate:wrapper:268 - ### ENTER find_path()
2026-01-15 14:01:05.392 | DEBUG    | passage.generate:find_path:527 - find_path: backend=networkx, solver=astar, diagonal=always, weight=30.0, heuristic=euclidean, cost_model=elevation, graph_backend=edges, grid_size=(128, 128), start=(np.int64(0), np.int64(17)), end=(np.int64(53), np.int64(69))
2026-01-15 14:01:05.392 | DEBUG    | passage.generate:wrapper:268 - ### ENTER build_graph_from_elevation()
2026-01-15 14:01:05.393 | DEBUG    | passage.generate:build_graph_from_elevation:446 - build_graph_from_elevation: start rows=128, cols=128, diagonal=always, cost_model=elevation, weight=30.0, graph_backend=edges



Running A* (Euclidean)...
  name=networkx:astar, cost_model=elevation


2026-01-15 14:01:05.736 | DEBUG    | passage.generate:build_graph_from_elevation:459 - build_graph_from_elevation: template fetch took 0.3426s
2026-01-15 14:01:05.738 | DEBUG    | passage.generate:build_graph_from_elevation:463 - build_graph_from_elevation: weights compute took 0.0018s
2026-01-15 14:01:05.983 | DEBUG    | passage.generate:build_graph_from_elevation:477 - build_graph_from_elevation: edge weights update took 0.2438s
2026-01-15 14:01:05.984 | DEBUG    | passage.generate:build_graph_from_elevation:478 - build_graph_from_elevation: total time 0.5911s
2026-01-15 14:01:05.984 | DEBUG    | passage.generate:wrapper:275 - ### EXIT build_graph_from_elevation() duration=0.5917s
2026-01-15 14:01:05.988 | DEBUG    | passage.generate:find_path:588 - Graph construction: nodes=16384, edges=129540, time=0.5925s
2026-01-15 14:01:06.032 | DEBUG    | passage.generate:find_path:630 - Pathfinding finished: path_len=67, graph_time=0.5925s, solve_time=0.0434s, total=0.6396s
2026-01-15 14:01:06

  Path length: 67 pixels
  Time: 642.0 ms

Running A* (Manhattan)...
  name=networkx:astar, cost_model=elevation


2026-01-15 14:01:06.279 | DEBUG    | passage.generate:build_graph_from_elevation:477 - build_graph_from_elevation: edge weights update took 0.2416s
2026-01-15 14:01:06.280 | DEBUG    | passage.generate:build_graph_from_elevation:478 - build_graph_from_elevation: total time 0.2458s
2026-01-15 14:01:06.281 | DEBUG    | passage.generate:wrapper:275 - ### EXIT build_graph_from_elevation() duration=0.2463s
2026-01-15 14:01:06.284 | DEBUG    | passage.generate:find_path:588 - Graph construction: nodes=16384, edges=129540, time=0.2471s
2026-01-15 14:01:06.327 | DEBUG    | passage.generate:find_path:630 - Pathfinding finished: path_len=67, graph_time=0.2471s, solve_time=0.0424s, total=0.2931s
2026-01-15 14:01:06.328 | DEBUG    | passage.generate:wrapper:275 - ### EXIT find_path() duration=0.2944s
2026-01-15 14:01:06.329 | DEBUG    | passage.generate:wrapper:268 - ### ENTER find_path()
2026-01-15 14:01:06.329 | DEBUG    | passage.generate:find_path:527 - find_path: backend=networkx, solver=dijk

  Path length: 67 pixels
  Time: 295.3 ms

Running Dijkstra...
  name=networkx:dijkstra, cost_model=elevation


... (4 outputs left out)

2026-01-15 14:01:07.159 | DEBUG    | passage.generate:build_graph_from_elevation:477 - build_graph_from_elevation: edge weights update took 0.2403s
2026-01-15 14:01:07.160 | DEBUG    | passage.generate:build_graph_from_elevation:478 - build_graph_from_elevation: total time 0.2442s
2026-01-15 14:01:07.161 | DEBUG    | passage.generate:wrapper:275 - ### EXIT build_graph_from_elevation() duration=0.2446s
2026-01-15 14:01:07.164 | DEBUG    | passage.generate:find_path:588 - Graph construction: nodes=16384, edges=129540, time=0.2454s
2026-01-15 14:01:07.323 | DEBUG    | passage.generate:find_path:630 - Pathfinding finished: path_len=67, graph_time=0.2454s, solve_time=0.1588s, total=0.4078s
2026-01-15 14:01:07.324 | DEBUG    | passage.generate:wrapper:275 - ### EXIT find_path() duration=0.4091s
2026-01-15 14:01:07.325 | DEBUG    | passage.generate:wrapper:268 - ### ENTER find_path()
2026-01-15 14:01:07.325 | DEBUG    | passage.generate:find_path:527 - find_path: backend=networkx, solver=unwe

  Path length: 67 pixels
  Time: 410.0 ms

Running Unweighted (BFS)...
  name=networkx:unweighted, cost_model=elevation


2026-01-15 14:01:07.572 | DEBUG    | passage.generate:build_graph_from_elevation:477 - build_graph_from_elevation: edge weights update took 0.2423s
2026-01-15 14:01:07.573 | DEBUG    | passage.generate:build_graph_from_elevation:478 - build_graph_from_elevation: total time 0.2465s
2026-01-15 14:01:07.573 | DEBUG    | passage.generate:wrapper:275 - ### EXIT build_graph_from_elevation() duration=0.2469s
2026-01-15 14:01:07.576 | DEBUG    | passage.generate:find_path:588 - Graph construction: nodes=16384, edges=129540, time=0.2477s
2026-01-15 14:01:07.584 | DEBUG    | passage.generate:find_path:630 - Pathfinding finished: path_len=54, graph_time=0.2477s, solve_time=0.0067s, total=0.2580s
2026-01-15 14:01:07.584 | DEBUG    | passage.generate:wrapper:275 - ### EXIT find_path() duration=0.2590s


  Path length: 54 pixels
  Time: 259.8 ms

Comparison complete!


## Results Summary

Compare all solvers in a table.

In [6]:
# Create results DataFrame
df = pd.DataFrame(
    [{"Solver": r["solver"], "Path Length": r["path_length"], "Time (ms)": f"{r['time_ms']:.1f}"} for r in results]
)

# Sort by time
df = df.sort_values("Time (ms)")
df = df.reset_index(drop=True)

print(f"Results for {RESOLUTION}x{RESOLUTION} resolution:")
print(f"Start: {start}, End: {end}")
print()
df

Results for 128x128 resolution:
Start: (np.int64(0), np.int64(17)), End: (np.int64(53), np.int64(69))



,Solver,Path Length,Time (ms)
0,Unweighted (BFS),54,259.8
1,Dijkstra (Bidirectional),67,286.0
2,A* (Manhattan),67,295.3
3,Dijkstra,67,300.1
4,Bellman-Ford,67,410.0
5,A* (Euclidean),67,642.0


## Timing Bar Chart

In [7]:
# Bar chart of execution times
fig, ax = plt.subplots(figsize=(10, 5))

solvers = [r["solver"] for r in results]
times = [r["time_ms"] for r in results]

bars = ax.barh(solvers, times, color="steelblue")
ax.set_xlabel("Time (ms)")
ax.set_title(f"Solver Comparison - {RESOLUTION}x{RESOLUTION} Resolution")
ax.invert_yaxis()

# Add time labels
for bar, t in zip(bars, times):
    ax.text(
        bar.get_width() + max(times) * 0.01, bar.get_y() + bar.get_height() / 2, f"{t:.1f} ms", va="center", fontsize=9
    )

plt.tight_layout()
plt.show()

## Path Visualization

Visualize paths from different solvers overlaid on the elevation map.

In [8]:
# Visualize paths
n_solvers = len(results)
cols = min(3, n_solvers)
rows = (n_solvers + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if n_solvers == 1:
    axes = np.array([axes])
axes = axes.flatten()

# Color map for elevation
for idx, (ax, result) in enumerate(zip(axes, results)):
    # Show elevation
    ax.imshow(elevation, cmap="terrain", alpha=0.7)

    # Draw path
    path = result["path"]
    if len(path) > 0:
        path_arr = np.array(path)
        ax.plot(path_arr[:, 1], path_arr[:, 0], "r-", linewidth=1, alpha=0.8, label="Path")

    # Mark start and end
    ax.scatter([start[1]], [start[0]], c="green", s=50, marker="o", label="Start", zorder=5)
    ax.scatter([end[1]], [end[0]], c="blue", s=50, marker="x", label="End", zorder=5)

    ax.set_title(f"{result['solver']}\n{result['time_ms']:.1f}ms, {result['path_length']} px")
    ax.axis("off")

# Hide unused axes
for ax in axes[n_solvers:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

## Path Comparison Detail

Compare path differences between solvers.

In [9]:
# Calculate path statistics
print("Path Statistics:")
print("=" * 60)

for result in results:
    path = result["path"]
    if len(path) > 1:
        # Calculate total path cost (sum of elevation along path)
        elevations = [elevation_raw[p[0], p[1]] for p in path]
        avg_elevation = np.mean(elevations)
        elevation_variance = np.var(elevations)

        # Calculate path straightness (euclidean / path length)
        euclidean = np.sqrt((start[0] - end[0]) ** 2 + (start[1] - end[1]) ** 2)
        straightness = euclidean / len(path) if len(path) > 0 else 0

        print(
            f"{result['solver']:25s} | Length: {len(path):5d} | "
            f"Avg Elev: {avg_elevation:7.1f}m | Straightness: {straightness:.3f}"
        )
    else:
        print(f"{result['solver']:25s} | No path found")

Path Statistics:
A* (Euclidean)            | Length:    67 | Avg Elev:  4582.5m | Straightness: 1.108
A* (Manhattan)            | Length:    67 | Avg Elev:  4582.5m | Straightness: 1.108
Dijkstra                  | Length:    67 | Avg Elev:  4582.5m | Straightness: 1.108
Dijkstra (Bidirectional)  | Length:    67 | Avg Elev:  4582.5m | Straightness: 1.108
Bellman-Ford              | Length:    67 | Avg Elev:  4582.5m | Straightness: 1.108
Unweighted (BFS)          | Length:    54 | Avg Elev:  4598.2m | Straightness: 1.375


## Recommendations

Based on the comparison:

- **A* with Euclidean heuristic**: Best overall balance of speed and optimality for 8-connectivity grids
- **A* with Manhattan heuristic**: Better for 4-connectivity (no diagonal moves)
- **Dijkstra**: Guaranteed optimal but slower than A* (no heuristic guidance)
- **Bellman-Ford**: Only needed for negative edge weights (slowest)
- **Unweighted (BFS)**: Fastest but ignores elevation costs (shortest path by number of steps)

Tip: try different `cost_model` settings to bias the path (e.g., `slope_uphill` to avoid climbing).